In [1]:
import pandas as pd 
df=pd.read_csv("PLT_NMS.txt")
df

,Symbol,Synonym Symbol,Scientific Name with Author,Common Name,Family
0,ABAB,NaN,Abutilon abutiloides (Jacq.) Garcke ex Hochr.,shrubby Indian mallow,Malvaceae
1,ABAB,ABAM5,Abutilon americanum (L.) Sweet,NaN,NaN
2,ABAB,ABJA,Abutilon jacquinii G. Don,NaN,NaN
3,ABAB,ABLI,Abutilon lignosum (Cav.) G. Don,NaN,NaN
4,ABAB70,NaN,Abietinella abietina (Hedw.) Fleisch.,abietinella moss,Thuidiaceae
...,...,...,...,...,...
93152,ZYVIR,ZYVIR2,Zygodon viridissimus (Dicks.) Brid. var. rufot...,NaN,NaN
93153,ZYVIR,ZYVIV2,Zygodon viridissimus (Dicks.) Brid. var. vulga...,NaN,NaN
93154,ZYVIR,ZYVU,Zygodon vulgaris (Malta) Nyholm,NaN,NaN
93155,ZYVIV,NaN,Zygodon viridissimus (Dicks.) Brid. var. virid...,zygodon moss,Orthotrichaceae


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93157 entries, 0 to 93156
Data columns (total 5 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   Symbol                       93157 non-null  object
 1   Synonym Symbol               44163 non-null  object
 2   Scientific Name with Author  93157 non-null  object
 3   Common Name                  43776 non-null  object
 4   Family                       48994 non-null  object
dtypes: object(5)
memory usage: 3.6+ MB


In [3]:
df.isna().sum()

Symbol                             0
Synonym Symbol                 48994
Scientific Name with Author        0
Common Name                    49381
Family                         44163
dtype: int64

In [4]:
df.dropna()

,Symbol,Synonym Symbol,Scientific Name with Author,Common Name,Family


In [5]:
df1=df["Scientific Name with Author"]
df1

0            Abutilon abutiloides (Jacq.) Garcke ex Hochr.
1                           Abutilon americanum (L.) Sweet
2                                Abutilon jacquinii G. Don
3                          Abutilon lignosum (Cav.) G. Don
4                    Abietinella abietina (Hedw.) Fleisch.
                               ...                        
93152    Zygodon viridissimus (Dicks.) Brid. var. rufot...
93153    Zygodon viridissimus (Dicks.) Brid. var. vulga...
93154                      Zygodon vulgaris (Malta) Nyholm
93155    Zygodon viridissimus (Dicks.) Brid. var. virid...
93156    Zygodon viridissimus (Dicks.) Brid. var. denta...
Name: Scientific Name with Author, Length: 93157, dtype: object

In [6]:
df1.isna().sum()

np.int64(0)

In [7]:
df1.to_csv('plants_NM.txt', index=False)

In [8]:
df3=pd.read_csv("plants_NM.txt")
df3

,Scientific Name with Author
0,Abutilon abutiloides (Jacq.) Garcke ex Hochr.
1,Abutilon americanum (L.) Sweet
2,Abutilon jacquinii G. Don
3,Abutilon lignosum (Cav.) G. Don
4,Abietinella abietina (Hedw.) Fleisch.
...,...
93152,Zygodon viridissimus (Dicks.) Brid. var. rufot...
93153,Zygodon viridissimus (Dicks.) Brid. var. vulga...
93154,Zygodon vulgaris (Malta) Nyholm
93155,Zygodon viridissimus (Dicks.) Brid. var. virid...


In [9]:
import pandas as pd
import re
from pathlib import Path

# --- Load and prepare data ---
in_path = Path("PLT_NMS.txt")
df = pd.read_csv(in_path)

# Limit to top 20,000 rows (or fewer if file is shorter)
n = min(20000, len(df))
df_top20k = df.head(n).copy()

# Add empty columns for enrichment
df_top20k["Medicine Product"] = ""
df_top20k["Cure Disease"] = ""

# --- Mapping dictionary ---
mapping = {
    "aloe": ("Aloe vera gel/extract", "Burns, wound healing, minor skin irritations; (oral: digestive issues) — traditional/clinical"),
    "allium sativum": ("Garlic extract/garlic oil", "Cardiovascular support; antimicrobial — traditional/clinical"),
    "curcuma longa": ("Turmeric/curcumin extract", "Anti-inflammatory; osteoarthritis — traditional/clinical"),
    "zingiber officinale": ("Ginger extract/tea", "Nausea, indigestion, motion sickness — traditional/clinical"),
    "mentha piperita": ("Peppermint oil/tea", "IBS symptoms, indigestion — traditional/clinical"),
    "eucalyptus": ("Eucalyptus oil", "Topical decongestant, respiratory symptom relief — traditional"),
    "echinacea": ("Echinacea extract", "Immune support, cold symptom relief — traditional"),
    "chamomile": ("Chamomile tea/extract", "Mild insomnia, digestive upset — traditional"),
    "calendula": ("Calendula extract", "Wound healing, skin irritations — traditional"),
    "ginkgo biloba": ("Ginkgo biloba extract", "Cognitive support, circulation — traditional/clinical"),
    "panax ginseng": ("Ginseng extract", "Adaptogen: energy, cognitive support — traditional"),
    "hypericum perforatum": ("St John's wort extract", "Mild depression — clinical/traditional"),
    "salix": ("Willow bark extract", "Pain relief, anti-inflammatory — traditional"),
    "arnica montana": ("Arnica topical", "Bruises, sprains, muscle pain — traditional"),
    "rosmarinus officinalis": ("Rosemary extract", "Cognitive alertness, digestive aid — traditional"),
    "thymus vulgaris": ("Thyme extract", "Antimicrobial, respiratory uses — traditional"),
    "glycyrrhiza glabra": ("Licorice root extract", "Ulcerative conditions, cough; expectorant — traditional"),
    "taraxacum officinale": ("Dandelion extract", "Digestive aid, diuretic — traditional"),
    "centella asiatica": ("Gotu kola extract", "Wound healing, cognitive support — traditional"),
    "sambucus nigra": ("Elderberry syrup", "Cold and flu symptom relief — traditional"),
    "silybum marianum": ("Milk thistle extract", "Liver support, hepatic protection — traditional/clinical"),
    "cinnamomum verum": ("Cinnamon extract", "Blood glucose modulation — traditional"),
    "syzygium aromaticum": ("Clove oil", "Toothache relief, topical antiseptic — traditional"),
    "trigonella foenum-graecum": ("Fenugreek extract", "Lactation support, glucose modulation — traditional"),
    "withania somnifera": ("Ashwagandha extract", "Adaptogen: stress, anxiety, fatigue — traditional"),
    "azadirachta indica": ("Neem extract", "Topical antiseptic, antiparasitic — traditional"),
    "nigella sativa": ("Black seed oil", "Anti-inflammatory, respiratory uses — traditional"),
    "salvia officinalis": ("Sage extract", "Sore throat gargle, cognitive support — traditional"),
    "lavandula angustifolia": ("Lavender oil", "Anxiety, mild insomnia — traditional"),
    "valeriana officinalis": ("Valerian extract", "Insomnia, mild anxiety — traditional"),
    "crataegus monogyna": ("Hawthorn extract", "Heart health, circulation — traditional"),
    "camellia sinensis": ("Green/black tea", "Antioxidant, mild stimulant"),
    
    "aloe": ("Aloe vera gel/extract", "Burns, wound healing, minor skin irritations; (oral: digestive issues)"),
    "garlic": ("Garlic extract/garlic oil", "Cardiovascular health (cholesterol, blood pressure), antimicrobial"),
    "turmeric": ("Curcumin/turmeric extract", "Anti-inflammatory uses, osteoarthritis, digestive issues"),
    "ginger": ("Ginger extract/tea", "Nausea, indigestion, motion sickness; anti-inflammatory"),
    "peppermint": ("Peppermint oil/tea", "Irritable bowel syndrome (IBS) symptoms, indigestion, topical analgesic"),
    "holy basil": ("Tulsi extract/tea", "Respiratory ailments, adaptogenic uses for stress and immunity"),
    "eucalyptus": ("Eucalyptus oil", "Topical decongestant (respiratory), topical antiseptic"),
    "echinacea": ("Echinacea extracts", "Immune support, common cold symptom relief (traditional use)"),
    "chamomile": ("Chamomile tea/extract", "Mild insomnia, digestive upset, anti-inflammatory (topical)"),
    "calendula": ("Calendula ointment/extract", "Wound healing, skin irritations, minor burns"),
    "ginkgo": ("Ginkgo biloba extract", "Cognitive support, peripheral circulation (traditional/use)"),
    "ginseng": ("Panax ginseng extract", "Energy, cognitive support, adaptogen"),
    "st john's wort": ("St John's wort extract", "Mild-to-moderate depression (traditional/clinical)"),
    "willow": ("Willow bark extract (salicylates)", "Pain relief, anti-inflammatory (historical source of aspirin)"),
    "arnica": ("Arnica topical ointment", "Topical treatment for bruises, sprains, muscle pain"),
    "rosemary": ("Rosemary oil/extract", "Cognitive alertness, digestive aid (traditional)"),
    "thyme": ("Thyme oil/extract", "Antimicrobial, respiratory uses"),
    "licorice": ("Licorice root extract (deglycyrrhizinated forms used)", "Ulcerative conditions, cough; expectorant (traditional)"),
    "dandelion": ("Dandelion extracts/tea", "Digestive aid, diuretic, liver support (traditional)"),
    "gotu kola": ("Centella asiatica extract", "Wound healing, cognitive support, circulatory issues (venous insufficiency)"),
    "elderberry": ("Elderberry syrup/extract", "Cold and influenza symptom relief (traditional)"),
    "milk thistle": ("Silymarin (milk thistle extract)", "Liver support, hepatic protection (traditional/clinical)"),
    "cinnamon": ("Cinnamon bark/extract", "Digestive aid, blood glucose modulation (traditional)"),
    "clove": ("Clove oil (eugenol)", "Toothache relief, topical antiseptic"),
    "fenugreek": ("Fenugreek seed extract", "Lactation support, blood glucose modulation"),
    "ashwagandha": ("Withania somnifera extract", "Adaptogen: stress, anxiety, fatigue"),
    "neem": ("Neem oil/extract", "Topical antiseptic, antiparasitic, dental hygiene"),
    "black seed": ("Nigella sativa oil", "Traditional uses: anti-inflammatory, respiratory conditions"),
    "sage": ("Sage extract/oil", "Sore throat gargle, cognitive support (traditional)"),
    "rosemary": ("Rosemary oil/extract", "Cognitive/antioxidant uses"),
     "Acorus calamus": ("Calamus oil/extract", "Digestive aid; respiratory issues (traditional)"),
    "Aloe vera": ("Aloe vera gel/extract", "Burns, wound healing, skin irritations; digestive uses"),
    "Allium sativum": ("Garlic extract/garlic oil", "Cardiovascular support; antimicrobial"),
    "Curcuma longa": ("Turmeric/curcumin extract", "Anti-inflammatory; osteoarthritis"),
    "Zingiber officinale": ("Ginger extract/tea", "Nausea, digestion, anti-inflammatory"),
    "Mentha piperita": ("Peppermint oil/tea", "IBS symptoms, indigestion"),
    "Eucalyptus globulus": ("Eucalyptus oil", "Topical decongestant, respiratory relief"),
    "Echinacea purpurea": ("Echinacea extract/syrup", "Immune support; common cold (traditional)"),
    "Matricaria chamomilla": ("Chamomile tea/extract", "Mild insomnia, digestive upset"),
    "Calendula officinalis": ("Calendula ointment/extract", "Wound healing, skin irritations"),
    "Ginkgo biloba": ("Ginkgo biloba extract", "Cognitive support, circulation"),
    "Panax ginseng": ("Ginseng extract", "Adaptogen; energy/cognitive support"),
    "Hypericum perforatum": ("St John's wort extract", "Mild-to-moderate depression"),
    "Salix alba": ("Willow bark extract (salicylates)", "Pain relief, anti-inflammatory"),
    "Arnica montana": ("Arnica topical ointment", "Bruises, sprains, muscle pain (topical)"),
    "Rosmarinus officinalis": ("Rosemary oil/extract", "Digestive aid; cognitive alertness"),
    "Thymus vulgaris": ("Thyme oil/extract", "Antimicrobial; respiratory uses"),
    "Glycyrrhiza glabra": ("Licorice root extract", "Ulcerative conditions; cough (traditional)"),
    "Taraxacum officinale": ("Dandelion extract/tea", "Digestive aid; diuretic"),
    "Centella asiatica": ("Gotu kola extract", "Wound healing; cognitive support"),
    "Sambucus nigra": ("Elderberry syrup/extract", "Cold/flu symptom relief (traditional)"),
    "Silybum marianum": ("Silymarin (milk thistle extract)", "Liver support, hepatic protection"),
    "Cinnamomum verum": ("Cinnamon bark/extract", "Digestive aid; blood glucose modulation (traditional)"),
    "Syzygium aromaticum": ("Clove oil (eugenol)", "Toothache relief; topical antiseptic"),
    "Trigonella foenum-graecum": ("Fenugreek seed extract", "Lactation support; blood glucose modulation"),
    "Withania somnifera": ("Ashwagandha extract", "Adaptogen: stress, anxiety"),
    "Azadirachta indica": ("Neem oil/extract", "Topical antiseptic; antiparasitic"),
    "Nigella sativa": ("Black seed oil (Nigella sativa)", "Anti-inflammatory; respiratory uses (traditional)"),
    "Salvia officinalis": ("Sage extract/oil", "Sore throat gargle; cognitive support"),
    "Lavandula angustifolia": ("Lavender oil/extract", "Anxiety, mild insomnia"),
    "Origanum vulgare": ("Oregano oil/extract", "Antimicrobial (traditional)"),
    "Valeriana officinalis": ("Valerian root extract", "Insomnia, mild anxiety"),
    "Passiflora incarnata": ("Passionflower extract", "Anxiety, insomnia (traditional)"),
    "Crataegus monogyna": ("Hawthorn extract", "Circulatory support; mild heart failure"),
    "Plantago major": ("Plantain poultice/extract", "Wound healing, skin irritations"),
    "Achillea millefolium": ("Yarrow extract", "Wound healing, bleeding control (traditional)"),
    "Capsicum annuum": ("Capsaicin topical", "Topical analgesic for neuropathic pain"),
    "Melaleuca alternifolia": ("Tea tree oil", "Topical antiseptic, antifungal"),
    "Hydrastis canadensis": ("Goldenseal extract", "Traditional antimicrobial (limited clinical evidence)"),
    "Urtica dioica": ("Nettle extract", "Allergic rhinitis, diuretic (traditional)"),
    "Camellia sinensis": ("Green/black tea extracts", "Antioxidant, mild stimulant"),



}

# --- Helper for matching ---
lower_map = {k.lower(): v for k, v in mapping.items()}

def find_match(name):
    if not isinstance(name, str):
        return ("", "")
    n = name.lower()
    for key, val in lower_map.items():
        if key in n:
            return val
    words = re.split(r'[^a-zA-Z0-9]+', n)
    for w in words:
        if w in lower_map:
            return lower_map[w]
    return ("", "")

# --- Apply enrichment ---
filled = 0
for idx, row in df_top20k.iterrows():
    prod, cure = find_match(row.get("Scientific Name with Author", ""))
    if prod:
        df_top20k.at[idx, "Medicine Product"] = prod
        df_top20k.at[idx, "Cure Disease"] = cure
        filled += 1

# --- Display as DataFrame ---
enriched_df = df_top20k[df_top20k["Medicine Product"] != ""][["Scientific Name with Author", "Medicine Product", "Cure Disease"]]

print(f"✅ Processed {n} plant names")
print(f"✅ Filled {filled} entries with medicinal data")

enriched_df.tail(10)  # Display top 20 enriched rows


✅ Processed 20000 plant names
✅ Filled 192 entries with medicinal data


,Scientific Name with Author,Medicine Product,Cure Disease
13166,Campanula carnica Schiede ex Mert. & W.D.J. Ko...,Arnica topical ointment,"Topical treatment for bruises, sprains, muscle..."
14675,Calendula L.,Calendula ointment/extract,"Wound healing, skin irritations, minor burns"
15313,Calendula officinalis L.,Calendula ointment/extract,"Wound healing, skin irritations, minor burns"
15314,Calendula officinalis L. var. prolifera hort.,Calendula ointment/extract,"Wound healing, skin irritations, minor burns"
16223,Camellia sinensis (L.) Kuntze,Green/black tea extracts,"Antioxidant, mild stimulant"
16234,Camellia sinensis (L.) Kuntze var. assamica (J...,Green/black tea extracts,"Antioxidant, mild stimulant"
16244,Camellia sinensis (L.) Kuntze var. sinensis,Green/black tea extracts,"Antioxidant, mild stimulant"
16996,Centella asiatica (L.) Urb.,Gotu kola extract,Wound healing; cognitive support
17163,Centella asiatica auct. non (L.) Urb.,Gotu kola extract,Wound healing; cognitive support
19970,Cinnamomum verum J. Presl,Cinnamon bark/extract,Digestive aid; blood glucose modulation (tradi...


In [10]:
enriched_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 192 entries, 208 to 19970
Data columns (total 3 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   Scientific Name with Author  192 non-null    object
 1   Medicine Product             192 non-null    object
 2   Cure Disease                 192 non-null    object
dtypes: object(3)
memory usage: 6.0+ KB


In [12]:
enriched_df.to_csv('clean.csv', index=False)

In [13]:
import wikipediaapi
import pandas as pd

# Define the function to get Wikipedia info
def get_wikipedia_info(plant_name):
    wiki = wikipediaapi.Wikipedia('https://en.wikipedia.org/wiki/List_of_plants_used_in_herbalism#Z')  # English Wikipedia
    page = wiki.page(plant_name)
    
    if not page.exists():
        return "Page not found", "No medicinal information available"
    
    # Extract the main summary
    summary = page.summary[0:10]  # Limit to first 500 characters
    
    # Try to extract any section related to "Medicinal uses"
    uses = "Not specified"
    for section in page.sections:
        if "medicinal" in section.title.lower() or "uses" in section.title.lower():
            uses = section.text[0:10]  # Get a preview of medicinal section
            break

    return summary, uses


# List of plants to search
#plants=df3["Scientific Name with Author"]
plants = ["Burdock", "Curcuma longa", "Zingiber officinale"]
data = []

# Loop through plants and collect information
for plant in plants:
    summary, uses = get_wikipedia_info(plant)
    data.append({
        "Plant Name": plant,
        "Wikipedia Summary": summary,
        "Medicinal Uses": use
    })

# Create a DataFrame
df = pd.DataFrame(data)

# Display the first few rows
#print(df.head())
df

NameError: name 'use' is not defined

In [ ]:
import wikipediaapi
import pandas as pd
import re

# Function to get direct medicinal uses
def get_wikipedia_info(plant_name):
    wiki = wikipediaapi.Wikipedia('https://en.wikipedia.org/wiki/List_of_plants_used_in_herbalism#Z')
    page = wiki.page(plant_name)

    if not page.exists():
        return "Page not found", "No medicinal information available"

    # Combine text from the summary and any medicinal sections
    text = page.summary
    for section in page.sections:
        if "medicinal" in section.title.lower() or "uses" in section.title.lower():
            text += " " + section.text

    # Extract only sentences that mention medicinal or therapeutic use
    medicinal_sentences = []
    pattern = re.compile(
        r"([A-Z][^.]*?\b(use[sd]?|treats?|treatment|therapy|remedy|medicine|healing)\b[^.]*\.)",
        re.IGNORECASE
    )

    for match in pattern.findall(text):
        medicinal_sentences.append(match[0].strip())

    if not medicinal_sentences:
        medicinal_info = "No direct medicinal use found"
    else:
        medicinal_info = " ".join(medicinal_sentences)

    return page.summary[:30], medicinal_info[:100]  # Limit lengths for readability


# List of sample plants
plants=df3["Scientific Name with Author"]
#plants = ["Aloe vera", "Curcuma longa", "Zingiber officinale","Burdock","Neem"]
data = []

# Loop through plants and extract info
for plant in plants:
    summary, uses = get_wikipedia_info(plant)
    data.append({
        "Plant Name": plant,
        "Wikipedia Summary": summary,
        "Direct Medicinal Uses": uses
    })

# Create DataFrame
df = pd.DataFrame(data)
#print(df.head())
df
# Optional: Save to CSV
#df.to_csv("direct_medicinal_uses.csv", index=False)
#print("\n✅ Data saved to 'direct_medicinal_uses.csv'")


In [ ]:
# Step 1: Send a request to the target website
url = "https://quotes.toscrape.com/"
response = requests.get(url)

# Step 2: Parse the HTML content
soup = BeautifulSoup(response.text, "html.parser")

# Step 3: Extract specific data (e.g., quotes and authors)
quotes = soup.find_all("span", class_="text")
authors = soup.find_all("small", class_="author")

# Step 4: Display results
for quote, author in zip(quotes, authors):
 print(f"{quote.text} — {author.text}")

In [1]:
# Step 1: Install the library
!pip install simple_image_download

# Step 2: Import and create the downloader
from simple_image_download import simple_image_download as simp

response = simp.simple_image_download

# Step 3: Choose your keywords
keywords = [
    "blueberry leaf disease",
    "blueberry leaf rust",
    "blueberry leaf spot",
    "blueberry chlorosis",
    "blueberry powdery mildew"
]

# Step 4: Download images
for kw in keywords:
    response().download(kw, 30)  # downloads 30 images for each disease


  DEPRECATION: Building 'progressbar' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'progressbar'. Discussion can be found at https://github.com/pypa/pip/issues/6334


  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for progressbar: filename=progressbar-2.5-py3-none-any.whl size=12083 sha256=3b1ab2f1967b2ed12adf4b2c8e3f70610b91c9bba200a288c119918fb412cedc
  Stored in directory: c:\users\user\appdata\local\pip\cache\wheels\15\38\9d\0d335445b021c31266d906b07313cd961adc1e7cece18579e9
Successfully built progressbar

   ---------------------------------------- 0/3 [python-magic-bin]
   ---------------------------------------- 0/3 [python-magic-bin]
   ------------- -------------------------- 1/3 [progressbar]
   ------------- -------------------------- 1/3 [progressbar]
   ------------- -------------------------- 1/3 [progressbar]
   ------------- -------------------------- 1/3 [progressbar]
   ------------- -------------------------- 1/3 [progressbar]
   ------------- -------------------------- 1/3 [progressbar]
   -------------------------- ------------- 2/3 [simple_image_download]
  

AttributeError: module 'simple_image_download.simple_image_download' has no attribute 'simple_image_download'

In [7]:
!pip install icrawler

from icrawler.builtin import GoogleImageCrawler

# Create a crawler instance
crawler = GoogleImageCrawler(storage={'root_dir': 'blueberry_disease_images'})

# List of disease keywords
keywords = [
    "blueberry leaf disease",
    "blueberry leaf rust",
    "blueberry leaf spot",
    "blueberry powdery mildew"
]

# Download 30 images for each keyword

for kw in keywords:
    downloader.download(kw, 30)



   -------------------- ------------------- 1/2 [icrawler]
   -------------------- ------------------- 1/2 [icrawler]
   -------------------- ------------------- 1/2 [icrawler]
   -------------------- ------------------- 1/2 [icrawler]
   -------------------- ------------------- 1/2 [icrawler]
   -------------------- ------------------- 1/2 [icrawler]
   -------------------- ------------------- 1/2 [icrawler]
   ---------------------------------------- 2/2 [icrawler]



NameError: name 'downloader' is not defined

In [30]:
# Install icrawler if you haven't already
# !pip install icrawler

from icrawler.builtin import GoogleImageCrawler

# List of disease keywords
keywords = [
       
    "beans leaf spot",
    "beans bacterial blight",
    "beans anthracnose",
    "beans healthy leaf"

]

# Download images for each keyword
for kw in keywords:
    crawler = GoogleImageCrawler(storage={'root_dir': f'blueberry_disease_images/{kw}'})
    crawler.crawl(keyword=kw, max_num=200)


2025-10-27 13:26:15,241 - INFO - icrawler.crawler - start crawling...
2025-10-27 13:26:15,245 - INFO - icrawler.crawler - starting 1 feeder threads...
2025-10-27 13:26:15,253 - INFO - feeder - thread feeder-001 exit
2025-10-27 13:26:15,329 - INFO - icrawler.crawler - starting 1 parser threads...
2025-10-27 13:26:15,393 - INFO - icrawler.crawler - starting 1 downloader threads...
2025-10-27 13:26:17,548 - INFO - parser - parsing result page https://www.google.com/search?q=beans+leaf+spot&ijn=0&start=0&tbs=&tbm=isch
2025-10-27 13:26:17,842 - INFO - downloader - skip downloading file 000001.jpg
2025-10-27 13:26:17,849 - INFO - downloader - skip downloading file 000002.jpg
2025-10-27 13:26:17,852 - INFO - downloader - skip downloading file 000003.jpg
2025-10-27 13:26:17,854 - INFO - downloader - skip downloading file 000004.jpg
2025-10-27 13:26:17,856 - INFO - downloader - skip downloading file 000005.jpg
2025-10-27 13:26:17,861 - INFO - downloader - skip downloading file 000006.jpg
2025-1

In [ ]:
treatment_dict = {
    # Apple
    "apple apple scab": "Fungicide sprays (captan, myclobutanil)",
    "apple fire blight": "Prune infected branches + bactericides",
    "apple powdery mildew": "Sulfur-based fungicides",
    "apple healthy leaf": "No treatment needed",

    # Blueberry
    "blueberry leaf rust": "Mancozeb or systemic fungicide",
    "blueberry powdery mildew": "Sulfur-based fungicides",
    "blueberry leaf spot": "Copper-based fungicides",
    "blueberry healthy leaf": "No treatment needed",

    # Cherry
    "cherry leaf spot": "Copper-based fungicides",
    "cherry brown rot": "Fungicide sprays (captan or thiophanate-methyl)",
    "cherry healthy leaf": "No treatment needed",

    # Corn
    "corn northern leaf blight": "Fungicide sprays (azoxystrobin)",
    "corn gray leaf spot": "Fungicide application + crop rotation",
    "corn healthy leaf": "No treatment needed",

    # Grape
    "grape powdery mildew": "Sulfur-based fungicides",
    "grape downy mildew": "Copper-based sprays",
    "grape healthy leaf": "No treatment needed",

    # Orange
    "orange citrus greening": "Remove infected branches + zinc fertilizers",
    "orange leaf curl": "Copper fungicide sprays",
    "orange leaf spot": "Copper-based sprays",
    "orange healthy leaf": "No treatment needed",

    # Peach
    "peach leaf curl": "Fungicide sprays (mancozeb or copper)",
    "peach powdery mildew": "Sulfur-based fungicides",
    "peach bacterial spot": "Copper-based sprays",
    "peach healthy leaf": "No treatment needed",

    # Pepper
    "pepper bacterial spot": "Copper-based sprays",
    "pepper leaf curl": "Remove infected plants, use resistant varieties",
    "pepper healthy leaf": "No treatment needed",

    # Potato
    "potato late blight": "Fungicide sprays (mancozeb or chlorothalonil)",
    "potato early blight": "Fungicide sprays + crop rotation",
    "potato healthy leaf": "No treatment needed",

    # Raspberry
    "raspberry leaf spot": "Fungicide sprays (captan)",
    "raspberry cane blight": "Prune infected canes",
    "raspberry healthy leaf": "No treatment needed",

    # Soybean
    "soybean leaf spot": "Fungicide sprays (azoxystrobin)",
    "soybean bacterial blight": "Copper-based bactericides",
    "soybean rust": "Fungicide sprays (azoxystrobin)",
    "soybean healthy leaf": "No treatment needed",

    # Squash
    "squash powdery mildew": "Sulfur-based fungicides",
    "squash downy mildew": "Copper-based fungicide sprays",
    "squash healthy leaf": "No treatment needed",

    # Strawberry
    "strawberry leaf spot": "Fungicide sprays (captan)",
    "strawberry powdery mildew": "Sulfur-based fungicides",
    "strawberry healthy leaf": "No treatment needed",

    # Tomato
    "tomato early blight": "Fungicide sprays + crop rotation",
    "tomato late blight": "Fungicide sprays (mancozeb)",
    "tomato leaf curl": "Remove infected plants, use resistant varieties",
    "tomato healthy leaf": "No treatment needed",

    # Tobacco
    "tobacco mosaic virus": "Remove infected plants, use resistant varieties",
    "tobacco black shank": "Fungicide sprays (mefenoxam)",
    "tobacco healthy leaf": "No treatment needed",
    
    # Skumawiki (example placeholder)
    "skumawiki leaf disease": "Fungicide or pesticide sprays depending on infection",
    "skumawiki healthy leaf": "No treatment needed"
}


In [13]:
# Install icrawler if you haven't already
# !pip install icrawler

from icrawler.builtin import GoogleImageCrawler

# Dictionary of crops and their disease + healthy keywords
treatment_dict = {
    # Apple
    "apple apple scab": "Fungicide sprays (captan, myclobutanil)",
    "apple fire blight": "Prune infected branches + bactericides",
    "apple powdery mildew": "Sulfur-based fungicides",
    "apple healthy leaf": "No treatment needed",

    # Blueberry
    "blueberry leaf rust": "Mancozeb or systemic fungicide",
    "blueberry powdery mildew": "Sulfur-based fungicides",
    "blueberry leaf spot": "Copper-based fungicides",
    "blueberry healthy leaf": "No treatment needed",

    # Cherry
    "cherry leaf spot": "Copper-based fungicides",
    "cherry brown rot": "Fungicide sprays (captan or thiophanate-methyl)",
    "cherry healthy leaf": "No treatment needed",

    # Corn
    "corn northern leaf blight": "Fungicide sprays (azoxystrobin)",
    "corn gray leaf spot": "Fungicide application + crop rotation",
    "corn healthy leaf": "No treatment needed",

    # Grape
    "grape powdery mildew": "Sulfur-based fungicides",
    "grape downy mildew": "Copper-based sprays",
    "grape healthy leaf": "No treatment needed",

    # Orange
    "orange citrus greening": "Remove infected branches + zinc fertilizers",
    "orange leaf curl": "Copper fungicide sprays",
    "orange leaf spot": "Copper-based sprays",
    "orange healthy leaf": "No treatment needed",

    # Peach
    "peach leaf curl": "Fungicide sprays (mancozeb or copper)",
    "peach powdery mildew": "Sulfur-based fungicides",
    "peach bacterial spot": "Copper-based sprays",
    "peach healthy leaf": "No treatment needed",

    # Pepper
    "pepper bacterial spot": "Copper-based sprays",
    "pepper leaf curl": "Remove infected plants, use resistant varieties",
    "pepper healthy leaf": "No treatment needed",

    # Potato
    "potato late blight": "Fungicide sprays (mancozeb or chlorothalonil)",
    "potato early blight": "Fungicide sprays + crop rotation",
    "potato healthy leaf": "No treatment needed",

    # Raspberry
    "raspberry leaf spot": "Fungicide sprays (captan)",
    "raspberry cane blight": "Prune infected canes",
    "raspberry healthy leaf": "No treatment needed",

    # Soybean
    "soybean leaf spot": "Fungicide sprays (azoxystrobin)",
    "soybean bacterial blight": "Copper-based bactericides",
    "soybean rust": "Fungicide sprays (azoxystrobin)",
    "soybean healthy leaf": "No treatment needed",

    # Squash
    "squash powdery mildew": "Sulfur-based fungicides",
    "squash downy mildew": "Copper-based fungicide sprays",
    "squash healthy leaf": "No treatment needed",

    # Strawberry
    "strawberry leaf spot": "Fungicide sprays (captan)",
    "strawberry powdery mildew": "Sulfur-based fungicides",
    "strawberry healthy leaf": "No treatment needed",

    # Tomato
    "tomato early blight": "Fungicide sprays + crop rotation",
    "tomato late blight": "Fungicide sprays (mancozeb)",
    "tomato leaf curl": "Remove infected plants, use resistant varieties",
    "tomato healthy leaf": "No treatment needed",

    # Tobacco
    "tobacco mosaic virus": "Remove infected plants, use resistant varieties",
    "tobacco black shank": "Fungicide sprays (mefenoxam)",
    "tobacco healthy leaf": "No treatment needed",
    
    # Skumawiki (example placeholder)
    "skumawiki leaf disease": "Fungicide or pesticide sprays depending on infection",
    "skumawiki healthy leaf": "No treatment needed"
}


# Loop over each crop and download images
for crop, keywords in disease_keywords.items():
    for kw in keywords:
        print(f"Downloading images for: {kw}")
        crawler = GoogleImageCrawler(storage={'root_dir': f'{crop}_disease_images/{kw}'})
        crawler.crawl(keyword=kw, max_num=100)  # Adjust max_num as needed


NameError: name 'disease_keywords' is not defined

In [25]:

# Onion
treatment_dict.update({
    "onion purple blotch": "Fungicide sprays (chlorothalonil or mancozeb)",
    "onion downy mildew": "Remove infected leaves + apply fungicides (mancozeb or metalaxyl)",
    "onion thrips damage": "Insecticides (spinosad or neem oil) and cultural control",
    "onion white rot": "Fungicide soil treatment + crop rotation",
    "onion leaf disease": "Identify specific pathogen; apply corresponding fungicide or bactericide",
    "onion healthy leaf": "No treatment needed"
})

# Loop through treatment_dict and print disease name and treatment
print("Disease Name\t\t\tRecommended Treatment")
print("="*80)

for disease, treatment in treatment_dict.items():
    print(f"{disease:<100} {treatment}")


Disease Name			Recommended Treatment
apple apple scab                                                                                     Fungicide sprays (captan, myclobutanil)
apple fire blight                                                                                    Prune infected branches + bactericides
apple powdery mildew                                                                                 Sulfur-based fungicides
apple healthy leaf                                                                                   No treatment needed
blueberry leaf rust                                                                                  Mancozeb or systemic fungicide
blueberry powdery mildew                                                                             Sulfur-based fungicides
blueberry leaf spot                                                                                  Copper-based fungicides
blueberry healthy leaf                                

In [26]:
import csv

with open("plant_disease_treatments.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["Disease Name", "Recommended Treatment"])
    for disease, treatment in treatment_dict.items():
        writer.writerow([disease, treatment])

print("CSV file 'plant_disease_treatments.csv' created successfully!")


CSV file 'plant_disease_treatments.csv' created successfully!


In [27]:
import pandas as pd 
data=pd.read_csv("plant_disease_treatments.csv")
data

,Disease Name,Recommended Treatment
0,apple apple scab,"Fungicide sprays (captan, myclobutanil)"
1,apple fire blight,Prune infected branches + bactericides
2,apple powdery mildew,Sulfur-based fungicides
3,apple healthy leaf,No treatment needed
4,blueberry leaf rust,Mancozeb or systemic fungicide
5,blueberry powdery mildew,Sulfur-based fungicides
6,blueberry leaf spot,Copper-based fungicides
7,blueberry healthy leaf,No treatment needed
8,cherry leaf spot,Copper-based fungicides
9,cherry brown rot,Fungicide sprays (captan or thiophanate-methyl)


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm
import time

# Load plant names from the uploaded file
# Make sure PLT_NMS.txt is in the same directory as this script
df_plants = pd.read_csv("PLT_NMS.txt")

# Extract plant names from the 'Scientific Name with Author' column
# Drop missing entries and remove duplicates
plant_names = df_plants["Scientific Name with Author"].dropna().unique()

# Convert names into URL-friendly format (replace spaces with '+')
plants = [name.replace(" ", "+") for name in plant_names]

# Base URL of PFAF
base_url = "https://pfaf.org/user/Plant.aspx?LatinName="

results = []

for plant in tqdm(plants, desc="Scraping plants"):
    url = base_url + plant
    try:
        res = requests.get(url, timeout=10)
        soup = BeautifulSoup(res.text, 'html.parser')
        
        # Extract key sections
        name = soup.find("span", {"id": "ctl00_ContentPlaceHolder1_lblPlantName"})
        uses = soup.find("span", {"id": "ctl00_ContentPlaceHolder1_lblMedicinalUses"})
        habitat = soup.find("span", {"id": "ctl00_ContentPlaceHolder1_lblHabitat"})
        
        results.append({
            "Plant_Name": name.text.strip() if name else plant.replace("+", " "),
            "Medicinal_Uses": uses.text.strip() if uses else "",
            "Habitat": habitat.text.strip() if habitat else ""
        })
        
        # Pause between requests (to avoid being blocked)
        time.sleep(1)
    except Exception as e:
        print(f"Error scraping {plant}: {e}")
        continue

# Save results to CSV
df = pd.DataFrame(results)
df.to_csv("medicinal_plants_scraped.csv", index=False, encoding="utf-8")

print("✅ Data saved as 'medicinal_plants_scraped.csv'")


Scraping plants:   3%|▍             | 2830/93150 [1:57:29<128:40:37,  5.13s/it]

Error scraping Allium+schoenoprasum+L.+var.+laurentianum+Fernald: HTTPSConnectionPool(host='pfaf.org', port=443): Read timed out. (read timeout=10)


Scraping plants:   3%|▍             | 2831/93150 [1:57:40<169:29:07,  6.76s/it]

Error scraping Allium+schoenoprasum+L.+ssp.+sibiricum+(L.)+Celak.: HTTPSConnectionPool(host='pfaf.org', port=443): Read timed out. (read timeout=10)


Scraping plants:   3%|▍             | 3174/93150 [2:13:24<134:55:37,  5.40s/it]

Error scraping Rinodina+finkii+H.+Magn.: HTTPSConnectionPool(host='pfaf.org', port=443): Read timed out. (read timeout=10)


Scraping plants:   3%|▍             | 3175/93150 [2:13:34<172:20:53,  6.90s/it]

Error scraping Rinodina+inaequalis+H.+Magn.: HTTPSConnectionPool(host='pfaf.org', port=443): Read timed out. (read timeout=10)


Scraping plants:   4%|▌              | 3264/93150 [2:17:39<61:59:59,  2.48s/it]